In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from pathlib import Path

In [2]:
data_dir = Path("../data/survey_of_consumer_finance")
figs_dir = Path("../figs")
files = {2016: data_dir / "SCFP2016.csv",
         2019: data_dir / "SCFP2019.csv",
         2022: data_dir / "SCFP2022.csv"}
files



{2016: PosixPath('../data/survey_of_consumer_finance/SCFP2016.csv'),
 2019: PosixPath('../data/survey_of_consumer_finance/SCFP2019.csv'),
 2022: PosixPath('../data/survey_of_consumer_finance/SCFP2022.csv')}

In [3]:
# ---------------------------------------------------------------- settings
dollar_vars = ["WAGEINC", "LIQ", "NETWORTH"]
demo_vars = ["HHSEX", "AGE", "EDUC", "RACE"]
continuous_demo = ["AGE"]                       # demo vars treated as continuous
categorical_vars = [v for v in demo_vars if v not in continuous_demo]
continuous_vars = dollar_vars + continuous_demo
 
pcts = [1, 5, 10, 25, 50, 75, 90, 95, 99]
scale = 1_000
base_year = 2022
 
CPI = {2016: 240.007, 2019: 255.657, 2022: 292.655}
deflator = {y: CPI[base_year] / CPI[y] for y in CPI}
 
# The 2022 SCF split "Asian" (RACE code 4) out of "Other" (code 5); in 2016 and 2019 Asian
# households are inside "Other". True merges code 4 into code 5 so every wave is comparable;
# False shows Asian separately for 2022 only ("Other" is then not comparable across waves).
# The pooled sample requires True.
harmonize_race = True
 
race_labels = {1: "White non-Hispanic", 2: "Black/African American", 3: "Hispanic/Latino"}
race_labels.update({5: "Other (incl. Asian)"} if harmonize_race
                   else {4: "Asian", 5: "Other"})
race_note = ("Asian households are included in Other in all waves, since the 2022 survey "
             "first identified Asian separately." if harmonize_race else
             "Asian is identified separately only from the 2022 survey; in 2016 and 2019 "
             "Asian households are included in Other, so the Other category is not "
             "comparable across waves.")
 
# Optional labels; verify against each year's codebook. Unlisted codes print as numbers.
labels = {"HHSEX": {1: "Male", 2: "Female"}, "RACE": race_labels}
 
# ---------------------------------------------------------------- pooled sample
POOLED = "Pooled"
 
 
def _ysort(ys):
    """Numeric years ascending, Pooled last (mixed int/str labels cannot be sorted directly)."""
    ys = list(ys)
    return sorted(y for y in ys if y != POOLED) + ([POOLED] if POOLED in ys else [])
 
 
def _slug_label(x):
    """Safe LaTeX label fragment."""
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")
 
 
# ---------------------------------------------------------------- weighted moments
def wquantile(x, w, q):
    """Weighted quantile (inverse weighted CDF), q in [0, 1]."""
    idx = np.argsort(x, kind="mergesort")
    x, w = x[idx], w[idx]
    cw = np.cumsum(w) / w.sum()
    return x[np.minimum(np.searchsorted(cw, q), len(x) - 1)]
 
 
def wstats(x, w):
    """All statistics for one implicate; x, w are 1-d float arrays."""
    sw = w.sum()
    mu = (w * x).sum() / sw
    d = x - mu
    m2, m3, m4 = [(w * d**k).sum() / sw for k in (2, 3, 4)]
    out = {"Mean": mu, "Variance": m2, "Std. dev.": np.sqrt(m2),
           "Skewness": m3 / m2**1.5, "Kurtosis": m4 / m2**2,
           "Min": x.min(), "Max": x.max()}
    for p in pcts:
        out[f"P{p}"] = wquantile(x, w, p / 100)
    return out
 
 
# ---------------------------------------------------------------- one wave and pooled
def load_wave(path, year, harmonize=None):
    d = pd.read_csv(path)
    d.columns = d.columns.str.upper()
    d["YEAR"] = year
    d["IMP"] = d["Y1"] - 10 * d["YY1"]          # implicate 1..5
    d["W"] = d["WGT"] / 5                       # weights sum to population
    if harmonize_race if harmonize is None else harmonize:
        d["RACE"] = d["RACE"].replace(4, 5)     # fold Asian (2022 only) into Other
    for v in dollar_vars:
        d[v] = d[v] * deflator[year]
    return d
 
 
def load_all(files, pooled=True):
    """{year: frame} plus a Pooled frame (stacked waves, weights divided by number of waves).
    Dollar variables are deflated wave by wave before stacking."""
    waves = {yr: load_wave(p, yr) for yr, p in files.items()}
    if pooled:
        assert harmonize_race, ("Pooling requires harmonize_race = True "
                                "(Other is not comparable otherwise).")
        d = pd.concat(waves.values(), axis=0, ignore_index=True)   # unique index for add_groups
        d["W"] = d["W"] / len(waves)
        waves[POOLED] = d
    return waves
 
 
def summarize_continuous(d, variables=continuous_vars):
    """Statistics within each implicate, then averaged across the five implicates."""
    rows = {}
    for v in variables:
        per_imp = pd.DataFrame([wstats(g[v].to_numpy(float), g["W"].to_numpy(float))
                                for _, g in d.groupby("IMP")])
        s = per_imp.mean()
        if v in dollar_vars:                    # thousands of base-year dollars
            keep = ["Skewness", "Kurtosis"]
            s = s.copy()
            for k in s.index:
                if k == "Variance":
                    s[k] /= scale**2
                elif k not in keep:
                    s[k] /= scale
        rows[v] = s
    return pd.DataFrame(rows).T
 
 
def summarize_categorical(d, variables=categorical_vars):
    """Weighted population shares (percent) by category, averaged across implicates."""
    out = {}
    for v in variables:
        sh = pd.DataFrame({m: g.groupby(v)["W"].sum() / g["W"].sum() * 100
                           for m, g in d.groupby("IMP")}).fillna(0).mean(axis=1)
        sh.index = [labels.get(v, {}).get(int(c), int(c)) for c in sh.index]
        out[v] = sh
    return pd.concat(out, names=["Variable", "Category"])
 
 
def summarize_all(files, pooled=True):
    """files: {year: path}. Returns (continuous, categorical) long frames; Pooled column last."""
    cont, cat = {}, {}
    for yr, d in load_all(files, pooled).items():
        cont[yr] = summarize_continuous(d)
        cat[yr] = summarize_categorical(d)
    cont = pd.concat(cont, names=["Year", "Variable"]).swaplevel().loc[continuous_vars]
    latest = max(k for k in cat if k != POOLED)
    order = list(cat[latest].index)             # latest wave lists every category, in code order
    cat = pd.concat(cat, axis=1, sort=False)    # rows (Variable, Category); NaN = not in wave
    cat = cat.reindex(order + [i for i in cat.index if i not in order])
    cat.columns.name = "Year"
    return cont, cat
 
 
# ---------------------------------------------------------------- group comparisons
age_bins = [0, 35, 45, 55, 65, 75, 200]
age_labels = ["Under 35", "35-44", "45-54", "55-64", "65-74", "75 and over"]
edu_labels = ["Less than high school", "High school or GED",
              "Some college or associate", "Bachelor's degree or more"]
# EDUC codes 1-14 collapsed to four groups; verify against the codebook (cf. EDCL).
edu_map = {**{k: edu_labels[0] for k in range(1, 8)}, 8: edu_labels[1],
           **{k: edu_labels[2] for k in (9, 10, 11)},
           **{k: edu_labels[3] for k in (12, 13, 14)}}
 
group_order = {"Age of head": age_labels,
               "Education of head": edu_labels,
               "Race/ethnicity": list(labels["RACE"].values()),
               "Sex of head": list(labels["HHSEX"].values())}
 
 
def add_groups(d, race_map=None):
    g = pd.DataFrame(index=d.index)
    g["Age of head"] = pd.cut(d["AGE"], age_bins, labels=age_labels, right=False)
    g["Education of head"] = d["EDUC"].map(edu_map)
    g["Race/ethnicity"] = d["RACE"].map(race_map or labels["RACE"])
    g["Sex of head"] = d["HHSEX"].map(labels["HHSEX"])
    return g
 
 
def group_stats(d, variables=dollar_vars):
    """Weighted mean, median, variance, and std. dev. by group, averaged across implicates."""
    grp = add_groups(d)
    rec = []
    for m, g in d.groupby("IMP"):
        gg, w = grp.loc[g.index], g["W"].to_numpy(float)
        cells = [("All households", "All households", np.ones(len(g), bool))]
        cells += [(dim, lab, (gg[dim] == lab).to_numpy())
                  for dim, labs in group_order.items() for lab in labs]
        for dim, lab, mask in cells:
            if mask.sum() == 0:
                continue
            for v in variables:
                x, ww = g[v].to_numpy(float)[mask], w[mask]
                mu = np.average(x, weights=ww)
                var = np.average((x - mu) ** 2, weights=ww)
                rec.append((m, dim, lab, v, mu, wquantile(x, ww, 0.5), var, np.sqrt(var)))
    cols = ["Mean", "Median", "Variance", "Std. dev."]
    df = pd.DataFrame(rec, columns=["IMP", "Dimension", "Group", "Variable"] + cols)
    out = df.groupby(["Dimension", "Group", "Variable"])[cols].mean()
    out[["Mean", "Median", "Std. dev."]] /= scale      # thousands of base-year dollars
    out["Variance"] /= scale**2                        # squared thousands
    return out
 
 
def build_group_tables(files, pooled=True):
    """Long frame indexed by (Year, Dimension, Group, Variable); thousands of base-year $."""
    return pd.concat({yr: group_stats(d) for yr, d in load_all(files, pooled).items()},
                     names=["Year"])
 
 
# ---------------------------------------------------------------- LaTeX
def _tex(t, caption, label, notes, fmt="%.2f"):
    body = t.to_latex(float_format=fmt, na_rep="--", column_format="l" + "r" * t.shape[1])
    return (
        "\\begin{table}[htbp]\n\\centering\n"
        f"\\caption{{{caption}}}\n\\label{{{label}}}\n"
        "\\begin{threeparttable}\n"
        f"{body}"
        "\\begin{tablenotes}\n\\footnotesize\n"
        f"\\item \\textit{{Notes:}} {notes}\n"
        "\\end{tablenotes}\n\\end{threeparttable}\n\\end{table}\n"
    )
 
 
_src = ("Source: Federal Reserve Board, Survey of Consumer Finances Summary Extract. "
        "Statistics are weighted by the sampling weight, computed within each of the five "
        "implicates and averaged across implicates. Waves are independent cross-sections. "
        "The pooled sample stacks the 2016, 2019, and 2022 waves, with dollar values "
        "deflated wave by wave before stacking and weights divided by the number of waves.")
 
 
def latex_by_variable(cont, var):
    """Rows = statistics, columns = years (Pooled last), for one continuous variable."""
    if var in dollar_vars:
        unit = f"thousands of {base_year} dollars, except skewness and kurtosis"
        extra = (f" Dollar values are deflated to {base_year} dollars. Kurtosis is the "
                 "raw (non-excess) fourth standardized moment.")
    else:
        unit = "years, except skewness and kurtosis"
        extra = " Kurtosis is the raw (non-excess) fourth standardized moment."
    return _tex(cont.loc[var].T, f"Weighted summary statistics for {var} ({unit})",
                f"tab:scf_{var.lower()}", _src + extra)
 
 
def latex_by_year(cont, year):
    """Rows = variables, columns = statistics, for one year (or POOLED)."""
    cap = ("Weighted summary statistics, SCF 2016, 2019, and 2022 pooled" if year == POOLED
           else f"Weighted summary statistics, SCF {year}")
    return _tex(cont.xs(year, level="Year"), cap,
                f"tab:scf_{_slug_label(year)}",
                _src + f" Dollar variables are in thousands of {base_year} dollars; "
                "AGE is in years. Kurtosis is the raw fourth standardized moment.")
 
 
def latex_shares(cat, var):
    """Weighted population shares (percent) of a categorical variable, by year."""
    return _tex(cat.loc[var], f"Weighted population shares of {var} (percent)",
                f"tab:scf_{var.lower()}_shares",
                _src + (" " + race_note if var == "RACE" else ""), fmt="%.1f")
 
 
def latex_group_table(big, var, stats=("Mean", "Median", "Variance")):
    """Group statistics of `var`, one column block per year (Pooled last), panels by dimension.
    stats may be any of "Mean", "Median", "Variance", "Std. dev."."""
    t = big.xs(var, level="Variable")
    years = _ysort(t.index.get_level_values("Year").unique())
    k = len(stats)
    ncol = 1 + k * len(years)
 
    def cells(dim, lab):
        out = []
        for y in years:
            try:
                r = t.loc[(y, dim, lab)]
                out += [f"{r[c]:,.1f}" for c in stats]
            except KeyError:
                out += ["--"] * k
        return " & ".join(out)
 
    unit = (f"thousands of {base_year} dollars; variance in squared thousands"
            if "Variance" in stats else f"thousands of {base_year} dollars")
    L = ["\\begin{table}[htbp]", "\\centering",
         f"\\caption{{{var} by household characteristics ({unit})}}",
         f"\\label{{tab:scf_{var.lower()}_groups}}", "\\begin{threeparttable}",
         "\\begin{tabular}{l" + "r" * (k * len(years)) + "}", "\\toprule",
         " & " + " & ".join(f"\\multicolumn{{{k}}}{{c}}{{{y}}}" for y in years) + " \\\\"]
    L.append(" ".join(f"\\cmidrule(lr){{{2 + k*i}-{1 + k*(i+1)}}}" for i in range(len(years))))
    L.append("Group & " + " & ".join([" & ".join(stats)] * len(years)) + " \\\\")
    L += ["\\midrule", "All households & " + cells("All households", "All households") + " \\\\"]
    for dim, labs in group_order.items():
        L += ["\\midrule", f"\\multicolumn{{{ncol}}}{{l}}{{\\textit{{{dim}}}}} \\\\"]
        L += [f"\\hspace{{1em}}{lab} & " + cells(dim, lab) + " \\\\" for lab in labs]
    L += ["\\bottomrule", "\\end{tabular}", "\\begin{tablenotes}\\footnotesize",
          f"\\item \\textit{{Notes:}} {_src} Dollar values are deflated to {base_year} dollars. "
          "Variance is the weighted population variance within the group. "
          "Age, education, race/ethnicity, and sex refer to the head of the primary economic unit. "
          + race_note,
          "\\end{tablenotes}", "\\end{threeparttable}", "\\end{table}"]
    return "\n".join(L) + "\n"
 
 
# ---------------------------------------------------------------- two-way comparisons
def group_stats_2way(d, panel, row, variables=dollar_vars, race_map=None):
    """Weighted stats for each (panel group, row group) cell, averaged across implicates.
    panel, row: keys of group_order, e.g. "Race/ethnicity", "Age of head"."""
    grp = add_groups(d, race_map)
    order = dict(group_order)
    if race_map:
        order["Race/ethnicity"] = list(race_map.values())
    rec = []
    for m, g in d.groupby("IMP"):
        gg, w = grp.loc[g.index], g["W"].to_numpy(float)
        for pl in order[panel]:
            for rl in order[row]:
                mask = ((gg[panel] == pl) & (gg[row] == rl)).to_numpy()
                if mask.sum() == 0:
                    continue
                for v in variables:
                    x, ww = g[v].to_numpy(float)[mask], w[mask]
                    mu = np.average(x, weights=ww)
                    var = np.average((x - mu) ** 2, weights=ww)
                    rec.append((m, pl, rl, v, mu, wquantile(x, ww, 0.5), var,
                                np.sqrt(var), mask.sum(), 100 * w[mask].sum() / w.sum()))
    cols = ["Mean", "Median", "Variance", "Std. dev.", "N", "Share"]
    df = pd.DataFrame(rec, columns=["IMP", "Panel", "Row", "Variable"] + cols)
    out = df.groupby(["Panel", "Row", "Variable"])[cols].mean()
    out[["Mean", "Median", "Std. dev."]] /= scale
    out["Variance"] /= scale**2
    return out
 
 
def build_two_way_tables(files, panel, row, pooled=True):
    """Long frame indexed by (Year, Panel, Row, Variable). N is the unweighted cell count."""
    return pd.concat({yr: group_stats_2way(d, panel, row)
                      for yr, d in load_all(files, pooled).items()}, names=["Year"])
 
 
def latex_two_way_table(big2, var, panel, row, stats=("Mean", "Median", "Variance"), min_n=30):
    """`var` by `row` groups within each `panel` group (Pooled last). Cells with fewer than
    min_n unweighted observations are suppressed."""
    t = big2.xs(var, level="Variable")
    years = _ysort(t.index.get_level_values("Year").unique())
    k = len(stats)
    ncol = 1 + k * len(years)
 
    def cells(pl, rl):
        out = []
        for y in years:
            try:
                r = t.loc[(y, pl, rl)]
                out += [f"{r[c]:,.1f}" for c in stats] if r["N"] >= min_n else ["--"] * k
            except KeyError:
                out += ["--"] * k
        return " & ".join(out)
 
    unit = (f"thousands of {base_year} dollars; variance in squared thousands"
            if "Variance" in stats else f"thousands of {base_year} dollars")
    slug = lambda z: z.split("/")[0].split()[0].lower()
    L = ["\\begin{table}[htbp]", "\\centering",
         f"\\caption{{{var} by {row.lower()} within {panel.lower()} groups ({unit})}}",
         f"\\label{{tab:scf_{var.lower()}_{slug(panel)}_{slug(row)}}}", "\\begin{threeparttable}",
         "\\begin{tabular}{l" + "r" * (k * len(years)) + "}", "\\toprule",
         " & " + " & ".join(f"\\multicolumn{{{k}}}{{c}}{{{y}}}" for y in years) + " \\\\"]
    L.append(" ".join(f"\\cmidrule(lr){{{2 + k*i}-{1 + k*(i+1)}}}" for i in range(len(years))))
    L.append(f"{row} & " + " & ".join([" & ".join(stats)] * len(years)) + " \\\\")
    for pl in group_order[panel]:
        L += ["\\midrule", f"\\multicolumn{{{ncol}}}{{l}}{{\\textit{{{pl}}}}} \\\\"]
        L += [f"\\hspace{{1em}}{rl} & " + cells(pl, rl) + " \\\\" for rl in group_order[row]]
    L += ["\\bottomrule", "\\end{tabular}", "\\begin{tablenotes}\\footnotesize",
          f"\\item \\textit{{Notes:}} {_src} Dollar values are deflated to {base_year} dollars. "
          "Variance is the weighted population variance within the cell. "
          f"Cells with fewer than {min_n} unweighted observations are suppressed (--). "
          "Characteristics refer to the head of the primary economic unit. " + race_note,
          "\\end{tablenotes}", "\\end{threeparttable}", "\\end{table}"]
    return "\n".join(L) + "\n"
 
 
# ---------------------------------------------------------------- single-wave cross-sections
race_order_all = ["White non-Hispanic", "Black/African American", "Hispanic/Latino",
                  "Asian", "Other", "Other (incl. Asian)"]
 
 
def build_cross_section(path, year, panel, row):
    """Two-way statistics for ONE wave, using that wave's own race coding: Asian is a
    separate category in 2022 and is inside Other in 2016 and 2019."""
    d = load_wave(path, year, harmonize=False)
    rm = {1: "White non-Hispanic", 2: "Black/African American", 3: "Hispanic/Latino"}
    if (d["RACE"] == 4).any():
        rm.update({4: "Asian", 5: "Other"})
    else:
        rm[5] = "Other (incl. Asian)"
    return group_stats_2way(d, panel, row, race_map=rm)
 
 
def build_pooled_cross_section(files, panel, row):
    """Two-way statistics on the pooled sample, with harmonized race coding."""
    d = load_all(files, pooled=True)[POOLED]
    return group_stats_2way(d, panel, row)
 
 
def latex_cross_section_table(tab, panel, row, year, variables=dollar_vars,
                              stats=("Mean", "Median", "Variance"), show_share=True, min_n=30):
    """Single-sample table: `row` groups within each `panel` group, one column block per
    variable. `year` is a wave (2016, 2019, 2022) or POOLED. Cells with fewer than min_n
    unweighted observations are suppressed."""
    k, nv = len(stats), len(variables)
    present = lambda lvl: set(tab.index.get_level_values(lvl))
 
    def order(dim, lvl):
        base = race_order_all if dim == "Race/ethnicity" else group_order[dim]
        return [x for x in base if x in present(lvl)]
    panels, rows = order(panel, "Panel"), order(row, "Row")
    sh = 1 if show_share else 0
    ncol = 1 + sh + k * nv
 
    def cells(pl, rl):
        try:
            r0 = tab.loc[(pl, rl, variables[0])]
            if r0["N"] < min_n:
                raise KeyError
            out = [f"{r0['Share']:.1f}"] if show_share else []
            for v in variables:
                r = tab.loc[(pl, rl, v)]
                out += [f"{r[c]:,.1f}" for c in stats]
            return " & ".join(out)
        except KeyError:
            return " & ".join(["--"] * (sh + k * nv))
 
    pooled = year == POOLED
    asian = "Asian" in present("Panel") | present("Row")
    if pooled:
        note = ("Asian households are included in Other in all waves, since Asian was not "
                "identified separately before the 2022 survey.")
    elif asian:
        note = "Asian is identified separately beginning with the 2022 survey."
    else:
        note = ("Asian households are included in Other, since Asian was not identified "
                "separately before the 2022 survey.")
    sample = "SCF 2016, 2019, and 2022 pooled" if pooled else f"SCF {year}"
    slug = lambda z: z.split("/")[0].split()[0].lower()
    unit = (f"thousands of {base_year} dollars; variance in squared thousands"
            if "Variance" in stats else f"thousands of {base_year} dollars")
    L = ["\\begin{table}[htbp]", "\\centering",
         f"\\caption{{Weighted statistics by {panel.lower()} and {row.lower()}, {sample} ({unit})}}",
         f"\\label{{tab:scf_{_slug_label(year)}_{slug(panel)}_{slug(row)}}}",
         "\\begin{threeparttable}",
         "\\setlength{\\tabcolsep}{4pt}", "\\resizebox{\\textwidth}{!}{%",
         "\\begin{tabular}{l" + "r" * (sh + k * nv) + "}", "\\toprule",
         " & ".join([""] + [""] * sh + [f"\\multicolumn{{{k}}}{{c}}{{{v}}}" for v in variables]) + " \\\\"]
    L.append(" ".join(f"\\cmidrule(lr){{{2 + sh + k*i}-{1 + sh + k*(i+1)}}}" for i in range(nv)))
    L.append(" & ".join([row] + (["Share (\\%)"] if show_share else []) + list(stats) * nv) + " \\\\")
    for pl in panels:
        L += ["\\midrule", f"\\multicolumn{{{ncol}}}{{l}}{{\\textit{{{pl}}}}} \\\\"]
        L += [f"\\hspace{{1em}}{rl} & " + cells(pl, rl) + " \\\\" for rl in rows]
    L += ["\\bottomrule", "\\end{tabular}}", "\\begin{tablenotes}\\footnotesize",
          f"\\item \\textit{{Notes:}} {_src} Dollar values are deflated to {base_year} dollars. "
          "Share is the weighted percentage of all households in the cell. "
          "Variance is the weighted population variance within the cell. "
          f"Cells with fewer than {min_n} unweighted observations are suppressed (--). "
          "Characteristics refer to the head of the primary economic unit. " + note,
          "\\end{tablenotes}", "\\end{threeparttable}", "\\end{table}"]
    return "\n".join(L) + "\n"


In [4]:
cont, cat = summarize_all(files)

cont.loc["WAGEINC"].T                        # moments and percentiles by year
cat.loc["RACE"]          

/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["YEAR"] = year
/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["IMP"] = d["Y1"] - 10 * d["YY1"]          # implicate 1..5
/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many time

Year,2016,2019,2022,Pooled
Category,,,,
White non-Hispanic,68.034895,67.995674,69.887606,68.652175
Black/African American,15.854471,15.652299,12.963511,14.803481
Hispanic/Latino,11.336701,10.930385,11.094491,11.118856
Other (incl. Asian),4.773932,5.421641,6.054392,5.425488


In [5]:
print(latex_by_variable(cont, "NETWORTH"))

\begin{table}[htbp]
\centering
\caption{Weighted summary statistics for NETWORTH (thousands of 2022 dollars, except skewness and kurtosis)}
\label{tab:scf_networth}
\begin{threeparttable}
\begin{tabular}{lrrrr}
\toprule
Year & 2016 & 2019 & 2022 & Pooled \\
\midrule
Mean & 1036.96 & 990.99 & 1059.46 & 1029.29 \\
Variance & 65815914.80 & 55464233.89 & 65374104.19 & 62215963.87 \\
Std. dev. & 8111.17 & 7445.48 & 8085.29 & 7886.93 \\
Skewness & 90.67 & 89.61 & 92.76 & 91.38 \\
Kurtosis & 14532.85 & 13986.98 & 14517.07 & 14473.50 \\
Min & -3081.75 & -1205.92 & -479.22 & -3081.75 \\
Max & 2387195.80 & 2504595.25 & 2249004.74 & 2534358.36 \\
P1 & -120.71 & -124.64 & -76.47 & -107.07 \\
P5 & -25.89 & -24.38 & -9.88 & -18.57 \\
P10 & -1.54 & -0.60 & 0.44 & 0.00 \\
P25 & 15.28 & 16.50 & 27.02 & 18.87 \\
P50 & 146.34 & 161.24 & 192.08 & 165.27 \\
P75 & 556.50 & 535.48 & 658.34 & 583.37 \\
P90 & 1778.43 & 1618.81 & 1920.76 & 1755.36 \\
P95 & 3579.82 & 3435.45 & 3779.60 & 3596.95 \\
P99 & 15618.91

In [6]:
print(latex_by_variable(cont, "WAGEINC"))

\begin{table}[htbp]
\centering
\caption{Weighted summary statistics for WAGEINC (thousands of 2022 dollars, except skewness and kurtosis)}
\label{tab:scf_wageinc}
\begin{threeparttable}
\begin{tabular}{lrrrr}
\toprule
Year & 2016 & 2019 & 2022 & Pooled \\
\midrule
Mean & 92.71 & 86.18 & 83.34 & 87.35 \\
Variance & 126569.55 & 31303.64 & 46454.66 & 67572.34 \\
Std. dev. & 355.76 & 176.92 & 215.48 & 259.94 \\
Skewness & 96.94 & 24.02 & 25.20 & 88.57 \\
Kurtosis & 19364.48 & 3320.79 & 1399.44 & 22651.43 \\
Min & 0.00 & 0.00 & -0.00 & -0.00 \\
Max & 141458.65 & 60797.24 & 30133.66 & 141458.65 \\
P1 & 0.00 & 0.00 & 0.00 & 0.00 \\
P5 & 0.00 & 0.00 & 0.00 & 0.00 \\
P10 & 0.00 & 0.00 & 0.00 & 0.00 \\
P25 & 0.00 & 0.00 & 0.00 & 0.00 \\
P50 & 46.61 & 47.29 & 43.24 & 45.70 \\
P75 & 113.34 & 110.79 & 101.82 & 108.08 \\
P90 & 198.34 & 201.85 & 186.13 & 194.83 \\
P95 & 288.22 & 279.40 & 272.39 & 279.69 \\
P99 & 685.51 & 643.10 & 708.00 & 659.28 \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\footno

In [7]:
print(latex_by_variable(cont, "LIQ"))

\begin{table}[htbp]
\centering
\caption{Weighted summary statistics for LIQ (thousands of 2022 dollars, except skewness and kurtosis)}
\label{tab:scf_liq}
\begin{threeparttable}
\begin{tabular}{lrrrr}
\toprule
Year & 2016 & 2019 & 2022 & Pooled \\
\midrule
Mean & 58.97 & 54.20 & 61.55 & 58.26 \\
Variance & 185695.66 & 185697.50 & 391395.55 & 255692.28 \\
Std. dev. & 429.81 & 430.83 & 625.35 & 505.61 \\
Skewness & 60.35 & 171.77 & 218.56 & 189.85 \\
Kurtosis & 7218.23 & 52279.41 & 72597.17 & 69545.11 \\
Min & 0.00 & 0.00 & 0.00 & 0.00 \\
Max & 116849.17 & 163809.71 & 263290.60 & 263290.60 \\
P1 & 0.00 & 0.00 & 0.00 & 0.00 \\
P5 & 0.03 & 0.05 & 0.06 & 0.05 \\
P10 & 0.23 & 0.27 & 0.22 & 0.25 \\
P25 & 1.25 & 1.34 & 1.35 & 1.33 \\
P50 & 6.22 & 6.69 & 7.83 & 6.90 \\
P75 & 28.82 & 28.59 & 34.85 & 30.66 \\
P90 & 102.82 & 101.37 & 115.94 & 107.04 \\
P95 & 213.83 & 198.46 & 228.46 & 215.46 \\
P99 & 861.75 & 746.02 & 798.60 & 804.20 \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\footnotesize
\i

In [8]:
print(latex_by_variable(cont, "AGE"))

\begin{table}[htbp]
\centering
\caption{Weighted summary statistics for AGE (years, except skewness and kurtosis)}
\label{tab:scf_age}
\begin{threeparttable}
\begin{tabular}{lrrrr}
\toprule
Year & 2016 & 2019 & 2022 & Pooled \\
\midrule
Mean & 51.68 & 51.73 & 52.45 & 51.96 \\
Variance & 301.48 & 306.27 & 310.10 & 306.13 \\
Std. dev. & 17.36 & 17.50 & 17.61 & 17.50 \\
Skewness & 0.14 & 0.10 & 0.08 & 0.11 \\
Kurtosis & 2.15 & 2.08 & 2.05 & 2.09 \\
Min & 18.00 & 18.00 & 18.00 & 18.00 \\
Max & 95.00 & 95.00 & 95.00 & 95.00 \\
P1 & 21.00 & 21.00 & 21.00 & 21.00 \\
P5 & 25.00 & 25.00 & 25.00 & 25.00 \\
P10 & 28.00 & 29.00 & 29.00 & 29.00 \\
P25 & 37.00 & 37.00 & 38.00 & 37.00 \\
P50 & 52.00 & 52.00 & 52.00 & 52.00 \\
P75 & 65.00 & 65.00 & 66.00 & 65.00 \\
P90 & 76.00 & 75.00 & 76.00 & 76.00 \\
P95 & 81.00 & 80.00 & 80.00 & 81.00 \\
P99 & 89.00 & 88.00 & 89.00 & 89.00 \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\footnotesize
\item \textit{Notes:} Source: Federal Reserve Board, Survey of C

In [9]:
print(latex_shares(cat, "EDUC"))

\begin{table}[htbp]
\centering
\caption{Weighted population shares of EDUC (percent)}
\label{tab:scf_educ_shares}
\begin{threeparttable}
\begin{tabular}{lrrrr}
\toprule
Year & 2016 & 2019 & 2022 & Pooled \\
Category &  &  &  &  \\
\midrule
-1 & 0.3 & 0.2 & 0.3 & 0.3 \\
1 & 0.8 & 0.6 & 0.5 & 0.6 \\
2 & 1.4 & 0.8 & 0.9 & 1.0 \\
3 & 1.5 & 1.5 & 0.8 & 1.2 \\
4 & 1.7 & 1.6 & 1.4 & 1.6 \\
5 & 2.3 & 1.4 & 1.2 & 1.6 \\
6 & 2.2 & 2.0 & 1.7 & 2.0 \\
7 & 2.6 & 2.6 & 2.4 & 2.5 \\
8 & 26.0 & 24.5 & 23.7 & 24.7 \\
9 & 15.5 & 16.7 & 15.6 & 15.9 \\
10 & 6.3 & 6.2 & 6.5 & 6.3 \\
11 & 5.6 & 5.6 & 5.0 & 5.4 \\
12 & 20.9 & 21.1 & 24.1 & 22.1 \\
13 & 9.3 & 10.7 & 11.0 & 10.3 \\
14 & 3.8 & 4.5 & 5.0 & 4.4 \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\footnotesize
\item \textit{Notes:} Source: Federal Reserve Board, Survey of Consumer Finances Summary Extract. Statistics are weighted by the sampling weight, computed within each of the five implicates and averaged across implicates. Waves are independent c

In [10]:
print(latex_shares(cat, "HHSEX"))

\begin{table}[htbp]
\centering
\caption{Weighted population shares of HHSEX (percent)}
\label{tab:scf_hhsex_shares}
\begin{threeparttable}
\begin{tabular}{lrrrr}
\toprule
Year & 2016 & 2019 & 2022 & Pooled \\
Category &  &  &  &  \\
\midrule
Male & 72.5 & 73.0 & 72.1 & 72.6 \\
Female & 27.5 & 27.0 & 27.9 & 27.4 \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\footnotesize
\item \textit{Notes:} Source: Federal Reserve Board, Survey of Consumer Finances Summary Extract. Statistics are weighted by the sampling weight, computed within each of the five implicates and averaged across implicates. Waves are independent cross-sections. The pooled sample stacks the 2016, 2019, and 2022 waves, with dollar values deflated wave by wave before stacking and weights divided by the number of waves.
\end{tablenotes}
\end{threeparttable}
\end{table}



In [11]:
print(latex_shares(cat, "RACE"))

\begin{table}[htbp]
\centering
\caption{Weighted population shares of RACE (percent)}
\label{tab:scf_race_shares}
\begin{threeparttable}
\begin{tabular}{lrrrr}
\toprule
Year & 2016 & 2019 & 2022 & Pooled \\
Category &  &  &  &  \\
\midrule
White non-Hispanic & 68.0 & 68.0 & 69.9 & 68.7 \\
Black/African American & 15.9 & 15.7 & 13.0 & 14.8 \\
Hispanic/Latino & 11.3 & 10.9 & 11.1 & 11.1 \\
Other (incl. Asian) & 4.8 & 5.4 & 6.1 & 5.4 \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\footnotesize
\item \textit{Notes:} Source: Federal Reserve Board, Survey of Consumer Finances Summary Extract. Statistics are weighted by the sampling weight, computed within each of the five implicates and averaged across implicates. Waves are independent cross-sections. The pooled sample stacks the 2016, 2019, and 2022 waves, with dollar values deflated wave by wave before stacking and weights divided by the number of waves. Asian households are included in Other in all waves, since the 2022 survey first iden

In [12]:
big = build_group_tables(files) 

/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["YEAR"] = year
/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["IMP"] = d["Y1"] - 10 * d["YY1"]          # implicate 1..5
/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many time

In [13]:
print(latex_group_table(big, "WAGEINC"))

\begin{table}[htbp]
\centering
\caption{WAGEINC by household characteristics (thousands of 2022 dollars; variance in squared thousands)}
\label{tab:scf_wageinc_groups}
\begin{threeparttable}
\begin{tabular}{lrrrrrrrrrrrr}
\toprule
 & \multicolumn{3}{c}{2016} & \multicolumn{3}{c}{2019} & \multicolumn{3}{c}{2022} & \multicolumn{3}{c}{Pooled} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Group & Mean & Median & Variance & Mean & Median & Variance & Mean & Median & Variance & Mean & Median & Variance \\
\midrule
All households & 92.7 & 46.6 & 126,569.5 & 86.2 & 47.3 & 31,303.6 & 83.3 & 43.2 & 46,454.7 & 87.3 & 45.7 & 67,572.3 \\
\midrule
\multicolumn{13}{l}{\textit{Age of head}} \\
\hspace{1em}Under 35 & 76.0 & 56.4 & 26,822.2 & 72.5 & 57.0 & 4,039.7 & 72.4 & 53.0 & 6,746.4 & 73.6 & 55.0 & 12,338.3 \\
\hspace{1em}35-44 & 122.5 & 84.7 & 34,205.4 & 117.1 & 85.7 & 23,194.1 & 114.2 & 75.7 & 61,077.6 & 117.9 & 81.3 & 39,818.1 \\
\hspace{1em}45-54 & 132.4 & 85

In [14]:
print(latex_group_table(big, "LIQ"))

\begin{table}[htbp]
\centering
\caption{LIQ by household characteristics (thousands of 2022 dollars; variance in squared thousands)}
\label{tab:scf_liq_groups}
\begin{threeparttable}
\begin{tabular}{lrrrrrrrrrrrr}
\toprule
 & \multicolumn{3}{c}{2016} & \multicolumn{3}{c}{2019} & \multicolumn{3}{c}{2022} & \multicolumn{3}{c}{Pooled} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Group & Mean & Median & Variance & Mean & Median & Variance & Mean & Median & Variance & Mean & Median & Variance \\
\midrule
All households & 59.0 & 6.2 & 185,695.7 & 54.2 & 6.7 & 185,697.5 & 61.6 & 7.8 & 391,395.6 & 58.3 & 6.9 & 255,692.3 \\
\midrule
\multicolumn{13}{l}{\textit{Age of head}} \\
\hspace{1em}Under 35 & 14.1 & 3.7 & 4,960.5 & 14.6 & 4.1 & 1,957.1 & 20.2 & 5.3 & 3,597.7 & 16.3 & 4.2 & 3,487.7 \\
\hspace{1em}35-44 & 36.6 & 5.2 & 75,655.8 & 36.5 & 6.0 & 24,122.7 & 40.9 & 7.1 & 27,193.6 & 38.0 & 6.0 & 42,039.6 \\
\hspace{1em}45-54 & 60.5 & 6.0 & 314,828.0 & 62.9 & 8

In [15]:
print(latex_group_table(big, "NETWORTH"))

\begin{table}[htbp]
\centering
\caption{NETWORTH by household characteristics (thousands of 2022 dollars; variance in squared thousands)}
\label{tab:scf_networth_groups}
\begin{threeparttable}
\begin{tabular}{lrrrrrrrrrrrr}
\toprule
 & \multicolumn{3}{c}{2016} & \multicolumn{3}{c}{2019} & \multicolumn{3}{c}{2022} & \multicolumn{3}{c}{Pooled} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Group & Mean & Median & Variance & Mean & Median & Variance & Mean & Median & Variance & Mean & Median & Variance \\
\midrule
All households & 1,037.0 & 146.3 & 65,815,914.8 & 991.0 & 161.2 & 55,464,233.9 & 1,059.5 & 192.1 & 65,374,104.2 & 1,029.3 & 165.3 & 62,215,963.9 \\
\midrule
\multicolumn{13}{l}{\textit{Age of head}} \\
\hspace{1em}Under 35 & 114.6 & 16.5 & 818,913.3 & 101.3 & 18.5 & 431,672.7 & 183.4 & 39.1 & 719,227.6 & 133.1 & 22.3 & 654,738.1 \\
\hspace{1em}35-44 & 433.5 & 89.2 & 40,600,558.4 & 580.9 & 121.3 & 65,258,070.5 & 548.1 & 134.8 & 9,525,270.6 & 521

In [17]:
def pooled_subsample(files, col="HHSEX", value=1):
    """Pooled sample (deflated, race harmonized, weights / 3) restricted to col == value."""
    d = load_all(files, pooled=True)[POOLED]
    return d[d[col] == value].copy()


def _tag(tex, who):
    """Add the subsample to the caption and notes of a generated table."""
    tex = tex.replace("\\caption{", f"\\caption{{{who}: ", 1)
    return tex.replace("\\item \\textit{Notes:} ",
                       f"\\item \\textit{{Notes:}} Sample restricted to {who}. ", 1)


who = "male-headed households (HHSEX = 1), SCF 2016, 2019, and 2022 pooled"
dm = pooled_subsample(files)

# Table 1: full distribution statistics, one column per variable
cont_m = summarize_continuous(dm, dollar_vars)          # rows = variables, cols = statistics
print(_tag(_tex(cont_m.T,
                f"Weighted summary statistics (thousands of {base_year} dollars, "
                "except skewness and kurtosis)",
                "tab:scf_pooled_male_summary",
                _src + f" Dollar values are deflated to {base_year} dollars. "
                "Kurtosis is the raw (non-excess) fourth standardized moment."), who))

# Tables 2-4: mean, median, variance by household characteristics, one table per variable
big_m = pd.concat({POOLED: group_stats(dm)}, names=["Year"])
for v in dollar_vars:
    print(_tag(latex_group_table(big_m, v), who))

/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["YEAR"] = year
/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["IMP"] = d["Y1"] - 10 * d["YY1"]          # implicate 1..5
/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many time

\begin{table}[htbp]
\centering
\caption{male-headed households (HHSEX = 1), SCF 2016, 2019, and 2022 pooled: Weighted summary statistics (thousands of 2022 dollars, except skewness and kurtosis)}
\label{tab:scf_pooled_male_summary}
\begin{threeparttable}
\begin{tabular}{lrrr}
\toprule
 & WAGEINC & LIQ & NETWORTH \\
\midrule
Mean & 107.25 & 70.91 & 1283.80 \\
Variance & 90077.33 & 347576.98 & 81133954.76 \\
Std. dev. & 300.13 & 589.50 & 9006.47 \\
Skewness & 79.05 & 164.95 & 77.68 \\
Kurtosis & 17544.20 & 51850.78 & 10486.68 \\
Min & -0.00 & 0.00 & -1182.01 \\
Max & 141458.65 & 263290.60 & 2534358.36 \\
P1 & 0.00 & 0.00 & -98.01 \\
P5 & 0.00 & 0.09 & -11.36 \\
P10 & 0.00 & 0.40 & 1.37 \\
P25 & 1.46 & 2.01 & 35.38 \\
P50 & 62.21 & 9.62 & 223.29 \\
P75 & 129.71 & 38.79 & 755.59 \\
P90 & 228.50 & 126.24 & 2232.10 \\
P95 & 322.82 & 256.19 & 4555.18 \\
P99 & 821.15 & 961.30 & 17308.45 \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\footnotesize
\item \textit{Notes:} Sample restricted to mal

In [19]:
import re
dm = pooled_subsample(files)                       # pooled, HHSEX = 1 (defined earlier)
tab_m = group_stats_2way(dm, "Race/ethnicity", "Education of head")

tex = latex_cross_section_table(tab_m, "Race/ethnicity", "Education of head", POOLED)
tex = tex.replace("\\caption{", "\\caption{Male-headed households (HHSEX = 1): ", 1)
tex = tex.replace("\\item \\textit{Notes:} ",
                  "\\item \\textit{Notes:} Sample restricted to male-headed households "
                  "(HHSEX = 1). Share is the weighted percentage of all male-headed "
                  "households in the cell. ", 1)
print(tex)

/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["YEAR"] = year
/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["IMP"] = d["Y1"] - 10 * d["YY1"]          # implicate 1..5
/var/folders/9b/4f2bxvvd1mj1hdx98fkh1lcm0000gn/T/ipykernel_86251/3398636546.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many time

\begin{table}[htbp]
\centering
\caption{Male-headed households (HHSEX = 1): Weighted statistics by race/ethnicity and education of head, SCF 2016, 2019, and 2022 pooled (thousands of 2022 dollars; variance in squared thousands)}
\label{tab:scf_pooled_race_education}
\begin{threeparttable}
\setlength{\tabcolsep}{4pt}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lrrrrrrrrrr}
\toprule
 &  & \multicolumn{3}{c}{WAGEINC} & \multicolumn{3}{c}{LIQ} & \multicolumn{3}{c}{NETWORTH} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8} \cmidrule(lr){9-11}
Education of head & Share (\%) & Mean & Median & Variance & Mean & Median & Variance & Mean & Median & Variance \\
\midrule
\multicolumn{11}{l}{\textit{White non-Hispanic}} \\
\hspace{1em}Less than high school & 5.1 & 35.7 & 7.1 & 3,288.3 & 18.9 & 2.3 & 47,511.9 & 389.7 & 102.3 & 36,977,308.2 \\
\hspace{1em}High school or GED & 17.4 & 65.2 & 50.0 & 6,863.6 & 34.8 & 5.7 & 56,960.8 & 580.5 & 170.8 & 19,065,704.6 \\
\hspace{1em}Some college or associate & 18.4 & 